# Dispersion over time

**Dispersion** is the standard deviation (population, divide by N) of the single names' `all_upfront` (present value of expected loss, in points) across the index's priced names, one value per date.
Pick an index and optionally a date window and up to three other indices to overlay. The chart shades the dates on which pricing is sparse and marks each credit event (a name leaving the index);
below it, the names with sparse data and the dates with the lowest coverage are listed so you can see what the line rests on.

- Names that defaulted out of a series stay in it, at their original weight, until their index ex-date (`teams.structured_credit_abhutra.credit_events`).
- A name with no price on a date is left out of that date. The shaded bands are dates where the priced names carry less than `SPARSE_BELOW` of the live index weight.
- Everything is read from local files (`cache/`); see `CLAUDE.md` for how they are refreshed.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'analysis' / 'cdx_spreads.py').exists())     # project root, wherever the notebook is opened from
sys.path.insert(0, str(ROOT / 'analysis'))

import pandas as pd
import cdx_spreads as cs
import index_metrics as im
import charts
from IPython.display import HTML, display

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

show = cs.show_table      # no-wrap HTML table with fixed decimals
print(', '.join(cs.load_index_list()['index_name']))
cs.print_data_as_of()          # which published snapshot (data/) this notebook is reading


IG47, IG46, IG45, IG44, IG43, IG42, IG41, IG40, IG39, IG38, HY47, HY46, HY45, HY44, HY43, HY42, HY41, HY40, HY39, HY38, XO46, XO45, XO44, XO43, XO42, XO41, XO40, XO39, XO38, XO37, MA46, MA45, MA44, MA43, MA42, MA41, MA40, MA39, MA38, MA37
data published 2026-10-26 08:34; prices through 2026-10-23


## Parameters

In [2]:
INDEX = 'HY42'
COMPARE = []              # optional extra indices drawn as extra lines (up to 3), e.g. ['HY43', 'HY41']
START = None              # e.g. '2025-01-01'; None = from the index launch
END = None                # None = latest date
SPARSE_BELOW = 0.95       # shade dates where the priced names carry less than this share of the live index weight

## 1. Dispersion over time

In [3]:
def credit_events(index_name, first, last):
    """Names that defaulted out of the series inside the plotted window, as (ex-date, label) pairs for the chart markers."""
    if not im.EVENTS_FILE.exists():
        return []
    ev = pd.read_parquet(im.EVENTS_FILE)
    ev = ev[(ev['index_name'] == index_name) & (ev['index_ex_date'] >= first) & (ev['index_ex_date'] <= last)]
    return [(r.index_ex_date, f'{r.sn_ticker} left {index_name} on {r.index_ex_date:%d %b %Y} (credit event)') for r in ev.itertuples()]


el = im.name_el(INDEX)
met = im.index_metrics(INDEX, el=el)
if START:
    met = met[met['published_date'] >= pd.Timestamp(START)]
if END:
    met = met[met['published_date'] <= pd.Timestamp(END)]
met = met.reset_index(drop=True)

others = {k: im.index_metrics(k).set_index('published_date')['dispersion'] for k in COMPARE[:3]}
dates = sorted(set(met['published_date']).union(*[set(s.index) for s in others.values()])) if others else list(met['published_date'])
if others:
    dates = [d for d in dates if (not START or d >= pd.Timestamp(START)) and (not END or d <= pd.Timestamp(END))]
prim = met.set_index('published_date')
series = [dict(name=INDEX, values=prim['dispersion'].reindex(dates).values)]
for k, s in others.items():
    series.append(dict(name=k, values=s.reindex(dates).values))

sparse_flag = (prim['priced_weight'] < SPARSE_BELOW).reindex(dates).fillna(False).values
bands = charts.runs(dates, sparse_flag)
tip = [f"{int(prim.loc[d, 'n_priced'])} of {int(prim.loc[d, 'n_members'])} names priced ({prim.loc[d, 'priced_weight']:.0%} of weight)" if d in prim.index else '' for d in dates]
tab = {'priced names': [f"{int(prim.loc[d, 'n_priced'])} / {int(prim.loc[d, 'n_members'])}" if d in prim.index else '' for d in dates],
       'priced weight': [f"{prim.loc[d, 'priced_weight']:.1%}" if d in prim.index else '' for d in dates]}
events = credit_events(INDEX, dates[0], dates[-1])

last = met.iloc[-1]
lo, hi = met.loc[met['dispersion'].idxmin()], met.loc[met['dispersion'].idxmax()]
print(f"{INDEX}: {len(met):,} dates, {met['published_date'].iloc[0]:%Y-%m-%d} to {last['published_date']:%Y-%m-%d} | dispersion now {last['dispersion']:.2f} "
      f"(low {lo['dispersion']:.2f} on {lo['published_date']:%Y-%m-%d}, high {hi['dispersion']:.2f} on {hi['published_date']:%Y-%m-%d}) | "
      f"{(met['priced_weight'] < SPARSE_BELOW).sum()} of {len(met)} dates are sparse (priced weight < {SPARSE_BELOW:.0%}), {len(events)} credit events in the window")
display(HTML(charts.line_chart(
    dates, series, title=f'{INDEX} dispersion of single-name expected loss',
    subtitle='Standard deviation of all_upfront across priced names (points). Shaded: sparse pricing. Triangles: a name leaves the index after a credit event.',
    y_label='points', fmt=lambda v: f'{v:.2f}', bands=bands, band_label=f'Sparse pricing in {INDEX} (priced weight < {SPARSE_BELOW:.0%})' if len(bands) else '',
    events=events, event_label=f'Credit event in {INDEX}' if events else '', tip_extra=tip, table_extra=tab)))

HY42: 673 dates, 2024-03-27 to 2026-10-23 | dispersion now 13.64 (low 10.82 on 2026-05-06, high 16.53 on 2024-07-01) | 0 of 673 dates are sparse (priced weight < 95%), 2 credit events in the window


## 2. Names with sparse data
Every name that is not priced on all the days it was in the index, sparsest first. `longest gap` is the longest run of consecutive unpriced dates. Names whose Markit curve only starts later
(a renamed curve, a new issuer) or stops early show up here; `add-in days` counts the days priced from the Markit add-in instead of Databricks.

In [4]:
cov = im.name_coverage(INDEX, el)
sparse = cov[cov['priced_days'] < cov['live_days']].rename(columns={
    'member_name': 'name', 'live_days': 'live days', 'priced_days': 'priced days', 'priced_pct': 'priced %', 'first_priced': 'first priced',
    'last_priced': 'last priced', 'longest_gap': 'longest gap (days)', 'addin_days': 'add-in days'})
print(f'{len(sparse)} of {len(cov)} names have at least one unpriced day; {int((cov["priced_pct"] < 90).sum())} are priced on less than 90% of their days')
show(sparse, numeric={'priced %': 1})

11 of 100 names have at least one unpriced day; 3 are priced on less than 90% of their days


name,ticker,defaulted,live days,priced days,priced %,first priced,last priced,longest gap (days),add-in days
Sirius XM Radio LLC,SIRI,False,673,499,74.1,2024-11-26,2026-10-23,174,0
Expand Energy Corp,EXE,False,673,519,77.1,2024-10-29,2026-10-23,154,234
Vistance Networks Holdings LLC,COMM,False,673,536,79.6,2024-03-27,2026-04-16,136,264
DISH (defaulted),DISH,True,615,592,96.3,2024-03-27,2026-07-02,23,0
LINTA (defaulted),LINTA,True,556,541,97.3,2024-03-27,2026-04-22,15,541
American Airlines Group Inc,AAL,False,673,671,99.7,2024-03-27,2026-10-23,1,0
Carnival Corp Ltd,CCL,False,673,672,99.9,2024-03-27,2026-10-23,1,0
Community Health Systems Inc,CYH,False,673,672,99.9,2024-03-27,2026-10-23,1,0
Sabre Holdings Corp,SABHLD,False,673,672,99.9,2024-03-27,2026-10-23,1,0
Staples Inc,SPLS,False,673,672,99.9,2024-03-27,2026-10-23,1,0


## 3. Dates with the lowest coverage
The ten dates on which the priced names carry the smallest share of the live index weight, and who is missing on each.

In [5]:
worst = met.sort_values(['priced_weight', 'published_date']).head(10)
rows = []
for r in worst.itertuples():
    miss = im.missing_on(INDEX, r.published_date, el)
    rows.append({'date': r.published_date, 'live names': r.n_members, 'priced names': r.n_priced, 'priced weight (%)': r.priced_weight * 100,
                 'dispersion': r.dispersion, 'missing': ', '.join(miss[:8]) + (f' (+{len(miss) - 8} more)' if len(miss) > 8 else '')})
show(pd.DataFrame(rows), numeric={'priced weight (%)': 1, 'dispersion': 2})

date,live names,priced names,priced weight (%),dispersion,missing
2024-10-22,100,97,97.0,12.19,"Community Health Systems Inc, Expand Energy Corp, Sirius XM Radio LLC"
2026-10-09,98,96,98.0,13.23,"Sabre Holdings Corp, Vistance Networks Holdings LLC"
2026-06-02,99,97,98.0,12.14,"Carnival Corp Ltd, Vistance Networks Holdings LLC"
2026-07-03,99,97,98.0,12.74,"Vistance Networks Holdings LLC, DISH (defaulted)"
2026-07-06,99,97,98.0,12.69,"Vistance Networks Holdings LLC, DISH (defaulted)"
2026-07-07,99,97,98.0,12.73,"Vistance Networks Holdings LLC, DISH (defaulted)"
2026-07-08,99,97,98.0,12.92,"Vistance Networks Holdings LLC, DISH (defaulted)"
2026-07-09,99,97,98.0,12.98,"Vistance Networks Holdings LLC, DISH (defaulted)"
2026-07-10,99,97,98.0,13.05,"Vistance Networks Holdings LLC, DISH (defaulted)"
2026-07-13,99,97,98.0,13.30,"Vistance Networks Holdings LLC, DISH (defaulted)"
